In [4]:
import os
import warnings
warnings.filterwarnings("ignore")

import librosa
import librosa.display

import numpy as np
import pandas as pd

from tqdm import tqdm

from concurrent.futures import ProcessPoolExecutor

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [5]:
DATASET_PATH = r"D:\ML-PROJECTS\Emotion analysis\dataset\voice-Mozilla"

PROCESSED_PATH = os.path.join(DATASET_PATH, "processed")

FEATURE_PATH = r"D:\ML-PROJECTS\Emotion analysis\extracted_features"

CHECKPOINT_PATH = os.path.join(FEATURE_PATH, "checkpoints")

os.makedirs(FEATURE_PATH, exist_ok=True)
os.makedirs(CHECKPOINT_PATH, exist_ok=True)

print("Feature Folder :", FEATURE_PATH)

Feature Folder : D:\ML-PROJECTS\Emotion analysis\extracted_features


In [7]:
gender_df = pd.read_csv(
    os.path.join(PROCESSED_PATH, "gender_processed.csv")
)

age_df = pd.read_csv(
    os.path.join(PROCESSED_PATH, "age_processed.csv")
)

print("Gender Dataset :", gender_df.shape)
print("Age Dataset :", age_df.shape)

Gender Dataset : (73278, 9)
Age Dataset : (54593, 9)


In [8]:
SAMPLE_RATE = 16000

N_MFCC = 40

CHECKPOINT_INTERVAL = 2000

MAX_WORKERS = os.cpu_count()

print("CPU Workers :", MAX_WORKERS)

CPU Workers : 16


In [9]:
def add_statistics(feature_list, values):

    feature_list.append(np.mean(values))
    feature_list.append(np.std(values))

In [10]:
def generate_feature_names():

    columns = []

    base = [
        "zcr",
        "rms",
        "centroid",
        "bandwidth",
        "rolloff",
        "flatness"
    ]

    for item in base:

        columns.append(item+"_mean")
        columns.append(item+"_std")

    columns.append("tempo")

    for i in range(12):

        columns.append(f"chroma_{i+1}_mean")
        columns.append(f"chroma_{i+1}_std")

    for i in range(7):

        columns.append(f"contrast_{i+1}_mean")
        columns.append(f"contrast_{i+1}_std")

    for i in range(6):

        columns.append(f"tonnetz_{i+1}_mean")
        columns.append(f"tonnetz_{i+1}_std")

    for i in range(N_MFCC):

        columns.append(f"mfcc_{i+1}_mean")
        columns.append(f"mfcc_{i+1}_std")

    for i in range(N_MFCC):

        columns.append(f"delta_{i+1}_mean")
        columns.append(f"delta_{i+1}_std")

    for i in range(N_MFCC):

        columns.append(f"delta2_{i+1}_mean")
        columns.append(f"delta2_{i+1}_std")

    return columns

In [11]:
def validate_audio(y):

    if len(y) == 0:
        return False

    if np.max(np.abs(y)) == 0:
        return False

    if np.isnan(y).any():
        return False

    if np.isinf(y).any():
        return False

    return True

In [12]:
def extract_features(audio_path):

    y, sr = librosa.load(
        audio_path,
        sr=SAMPLE_RATE,
        mono=True
    )

    if not validate_audio(y):
        return None

    features = []

    # --------------------------------
    # Zero Crossing Rate
    # --------------------------------

    zcr = librosa.feature.zero_crossing_rate(y)

    add_statistics(features, zcr)

    # --------------------------------
    # RMS
    # --------------------------------

    rms = librosa.feature.rms(y=y)

    add_statistics(features, rms)

    # --------------------------------
    # Centroid
    # --------------------------------

    centroid = librosa.feature.spectral_centroid(
        y=y,
        sr=sr
    )

    add_statistics(features, centroid)

    # --------------------------------
    # Bandwidth
    # --------------------------------

    bandwidth = librosa.feature.spectral_bandwidth(
        y=y,
        sr=sr
    )

    add_statistics(features, bandwidth)

    # --------------------------------
    # Rolloff
    # --------------------------------

    rolloff = librosa.feature.spectral_rolloff(
        y=y,
        sr=sr
    )

    add_statistics(features, rolloff)

    # --------------------------------
    # Flatness
    # --------------------------------

    flatness = librosa.feature.spectral_flatness(
        y=y
    )

    add_statistics(features, flatness)

    # --------------------------------
    # Tempo
    # --------------------------------

    tempo = librosa.feature.tempo(
        y=y,
        sr=sr
    )[0]

    features.append(float(tempo))

    # --------------------------------
    # Chroma
    # --------------------------------

    chroma = librosa.feature.chroma_stft(
        y=y,
        sr=sr
    )

    for row in chroma:

        add_statistics(features, row)

    # --------------------------------
    # Contrast
    # --------------------------------

    contrast = librosa.feature.spectral_contrast(
        y=y,
        sr=sr
    )

    for row in contrast:

        add_statistics(features, row)

    # --------------------------------
    # Tonnetz
    # --------------------------------

    harmonic = librosa.effects.harmonic(y)

    tonnetz = librosa.feature.tonnetz(
        y=harmonic,
        sr=sr
    )

    for row in tonnetz:

        add_statistics(features, row)

    # --------------------------------
    # MFCC
    # --------------------------------

    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=N_MFCC
    )

    for row in mfcc:

        add_statistics(features, row)

    # --------------------------------
    # Delta
    # --------------------------------

    delta = librosa.feature.delta(mfcc)

    for row in delta:

        add_statistics(features, row)

    # --------------------------------
    # Delta²
    # --------------------------------

    delta2 = librosa.feature.delta(
        mfcc,
        order=2
    )

    for row in delta2:

        add_statistics(features, row)

    return features

In [13]:
sample_audio = gender_df.iloc[0]["processed_audio"]

features = extract_features(sample_audio)

feature_names = generate_feature_names()

print("Number of Features :", len(features))
print("Feature Names      :", len(feature_names))
print("Match              :", len(features) == len(feature_names))

Number of Features : 303
Feature Names      : 303
Match              : True


In [14]:
def process_audio(row, label_column):

    try:

        feature = extract_features(row["processed_audio"])

        if feature is None:

            return None

        feature.append(row[label_column])

        return feature

    except Exception as e:

        return {
            "file": row["processed_audio"],
            "error": str(e)
        }

In [15]:
from concurrent.futures import ProcessPoolExecutor
from concurrent.futures import as_completed

import time

In [29]:
def feature_extraction(
        dataframe,
        label_column,
        output_csv,
        failed_csv
):

    feature_names = generate_feature_names()
    feature_names.append(label_column)

    features = []
    failed = []

    start = time.time()

    total = len(dataframe)

    for idx, (_, row) in enumerate(
            tqdm(dataframe.iterrows(), total=total)
    ):

        try:

            result = process_audio(
                row,
                label_column
            )

            if result is None:
                continue

            if isinstance(result, dict):
                failed.append(result)
            else:
                features.append(result)

        except Exception as e:

            failed.append({

                "filename": row["processed_audio"],

                "error": str(e)

            })

        # -----------------------------
        # Checkpoint
        # -----------------------------

        if (idx + 1) % CHECKPOINT_INTERVAL == 0:

            checkpoint = pd.DataFrame(
                features,
                columns=feature_names
            )

            checkpoint.to_csv(

                os.path.join(

                    CHECKPOINT_PATH,

                    f"checkpoint_{idx+1}.csv"

                ),

                index=False

            )

            elapsed = time.time() - start

            speed = (idx + 1) / elapsed

            print("\n" + "="*60)

            print(f"Processed : {idx+1}/{total}")

            print(f"Speed : {speed:.2f} files/sec")

            print(f"Failed : {len(failed)}")

            print("Checkpoint Saved")

            print("="*60)

    features = pd.DataFrame(
        features,
        columns=feature_names
    )

    failed = pd.DataFrame(failed)

    features.to_csv(
        output_csv,
        index=False
    )

    failed.to_csv(
        failed_csv,
        index=False
    )

    return features, failed

In [30]:
gender_output = os.path.join(

    FEATURE_PATH,

    "gender_features.csv"

)

gender_failed = os.path.join(

    FEATURE_PATH,

    "failed_gender.csv"

)

gender_features, gender_failed_df = feature_extraction(
    dataframe=gender_df,
    label_column="gender",
    output_csv=gender_output,
    failed_csv=gender_failed
)

  3%|▎         | 2000/73278 [04:52<6:07:19,  3.23it/s]


Processed : 2000/73278
Speed : 6.84 files/sec
Failed : 0
Checkpoint Saved


  5%|▌         | 4001/73278 [09:39<6:27:51,  2.98it/s]


Processed : 4000/73278
Speed : 6.90 files/sec
Failed : 0
Checkpoint Saved


  8%|▊         | 6002/73278 [14:31<7:01:26,  2.66it/s] 

...

 96%|█████████▌| 70000/73278 [3:13:58<9:02:22,  9.93s/it]


Processed : 70000/73278
Speed : 6.01 files/sec
Failed : 0
Checkpoint Saved


 98%|█████████▊| 72000/73278 [3:22:26<2:57:42,  8.34s/it]


Processed : 72000/73278
Speed : 5.93 files/sec
Failed : 0
Checkpoint Saved


100%|██████████| 73278/73278 [3:27:10<00:00,  5.89it/s]  


In [32]:

print("Gender Features Shape")

print(gender_features.shape)

print()

print("Failed Files")

print(len(gender_failed_df))

Gender Features Shape
(73276, 304)

Failed Files
0


In [34]:
age_output = os.path.join(

    FEATURE_PATH,

    "age_features.csv"

)

age_failed = os.path.join(

    FEATURE_PATH,

    "failed_age.csv"

)

age_features, age_failed_df = feature_extraction(
    dataframe=age_df,
    label_column="age",
    output_csv=age_output,
    failed_csv=age_failed
)

  0%|          | 79/54593 [00:12<2:19:45,  6.50it/s]


KeyboardInterrupt: 